In [ ]:
import yfinance as yf
import pandas as pd
import warnings
from datetime import datetime

# Suppress warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

RAW_LIST = """
A, AA, AAL, AAP, AAPL, ABBV, ABNB, ABT, ACN, ADBE, ADI, ADM, ADP, ADSK, AEP, AES, AFL, AFRM, AGNC, AI, AIG, AKAM, ALB, ALK, ALL, AMAT, AMD, AMGN, AMRN, AMZN, APA, APH, APTV, AVGO, AXP, BA, BABA, BAC, BAX, BBY, BEN, BIDU, BIIB, BITO, BK, BKR, BMY, BP, BSX, BX, BYND, C, CAH, CAT, CB, CCI, CCJ, CCL, CF, CFG, CHWY, CI, CL, CLF, CLX, CMCSA, CME, CNC, CNP, COF, COIN, COP, COST, CPB, CPRI, CRM, CRON, CRWD, CSCO, CSX, CTVA, CVNA, CVS, CVX, CZR, D, DAL, DD, DE, DELL, DHI, DHR, DIA, DIS, DKNG, DLR, DOCU, DOW, DRI, DVN, DXC, EA, EBAY, ED, EEM, EFA, EIX, EL, EMR, ENPH, EOG, EQR, EQT, ETSY, EVRG, EW, EWJ, EWW, EWY, EWZ, EXC, EXPE, F, FANG, FAST, FCX, FDX, FE, FI, FITB, FOXA, FSLR, FTI, FTV, FXE, FXI, GD, GDX, GE, GILD, GLD, GLW, GM, GME, GOOG, GOOGL, GPRO, GPS, GS, HAL, HBAN, HBI, HCA, HD, HIG, HLT, HOG, HOLX, HON, HPE, HPQ, HYG, IBB, IBIT, IBM, ICE, IEF, INTC, IP, IPG, IRM, IVZ, IWM, IYR, JCI, JD, JETS, JNJ, JNPR, JPM, K, KHC, KMI, KO, KR, KRE, KSS, KWEB, LEN, LI, LKQ, LLY, LNC, LOW, LQD, LUMN, LUV, LVS, LYB, LYFT, MA, MAR, MARA, MAS, MCD, MCHP, MDLZ, MDT, MET, META, MGM, MMM, MO, MOS, MPC, MRK, MRNA, MRVL, MS, MSFT, MSTR, MTB, MTCH, MU, NCLH, NEE, NEM, NET, NFLX, NKE, NOW, NRG, NTAP, NTES, NVAX, NVDA, NWL, NWSA, O, OIH, OKE, OMC, ON, ORCL, OXY, PARA, PBR, PDD, PENN, PEP, PFE, PFG, PG, PGR, PINS, PLTR, PNC, PPL, PRU, PSX, PTON, PYPL, QCOM, QQQ, RBLX, RCL, RF, RIG, RIOT, RIVN, ROKU, ROST, RTX, RUN, SBUX, SCHD, SCHW, SEDG, SHOP, SIRI, SLB, SLV, SMCI, SMH, SNAP, SNOW, SO, SOFI, SOXL, SOXS, SPG, ^SPX, SPXL, SPXS, SPY, SQQQ, STX, SWKS, SYF, SYY, T, TAP, TBT, TCOM, TDOC, TFC, TGT, TJX, TLT, TMO, TMUS, TPR, TQQQ, TRIP, TRV, TSLA, TSM, TSN, TT, TTD, TTWO, TXN, U, UA, UAA, UAL, UBER, ULTA, UNG, UNH, UNP, UPS, UPST, URBN, USB, USO, UVXY, V, VALE, VFC, VGK, VTR, VXX, VZ, WAB, WBA, WBD, WDC, WFC, WM, WMB, WMT, WU, WY, WYNN, X, XBI, XEL, XHB, XLB, XLC, XLE, XLF, XLI, XLK, XLP, XLRE, XLU, XLV, XLY, XOM, XOP, XRT, XRX, ^XSP, XYZ, YELP, YINN, ZION, ZM
"""

def run_hybrid_strategy():
    black_list = ['X', 'PARA', 'FI', 'GPS', 'IPG', 'WBA', 'JNPR', 'HBI', 'K', 'HOLX', 'XYZ']
    symbols = [s.strip() for s in RAW_LIST.replace('\n', ',').split(',') if s.strip() and s.strip() not in black_list]
    
    print(f"Executing Market Scan for {len(symbols)} symbols...")
    all_data = yf.download(symbols, period="12d", interval="1d", group_by='ticker', progress=True, auto_adjust=True)
    
    buys, sells = [], []
    b_win, s_win = 0, 0
    MIN_GAP_PCT = 0.002 

    for ticker in symbols:
        try:
            if ticker not in all_data.columns.get_level_values(0): continue
            df = all_data[ticker].dropna()
            if len(df) < 6: continue

            c, c1, c2 = df.iloc[-1], df.iloc[-2], df.iloc[-3]
            avg_vol = df['Volume'].iloc[-7:-2].mean()
            if c1['Volume'] < (avg_vol * 0.8): continue
            if c['Close'] < 20: continue

            c2_body = abs(c2['Open'] - c2['Close'])
            c2_range = c2['High'] - c2['Low']
            if c2_range == 0 or (c2_body / c2_range) < 0.10: continue

            c1_open, c1_close, c1_high, c1_low = c1['Open'], c1['Close'], c1['High'], c1['Low']
            c1_body = abs(c1_open - c1_close)
            c1_upper_wick, c1_lower_wick = c1_high - max(c1_open, c1_close), min(c1_open, c1_close) - c1_low
            c1_is_green = c1_close > c1_open
            b_score, s_score = 0, 0

            gap_pct = (c['Open'] - c1_close) / c1_close
            if gap_pct >= MIN_GAP_PCT: b_score += 25
            elif gap_pct <= -MIN_GAP_PCT: s_score += 25

            for wick in [c1_upper_wick, c1_lower_wick]:
                if wick < c1_body: # Continuation
                    if c1_is_green: b_score += 25
                    else: s_score += 25
                else: # Reversal
                    if c1_is_green: s_score += 25
                    else: b_score += 25

            if c1_body > (0.5 * abs(c2['Open'] - c2['Close'])): # Continuation
                if c1_is_green: b_score += 25
                else: s_score += 25
            else: # Reversal
                if c1_is_green: s_score += 25
                else: b_score += 25

            if b_score == 100: buys.append(ticker)
            elif s_score == 100: sells.append(ticker)
            if b_score == 100 and c['Close'] > c['Open']: b_win += 1
            if s_score == 100 and c['Close'] < c['Open']: s_win += 1
        except: continue

    as_of = all_data.index[-1].date()
    pd.DataFrame({'Date': [as_of], 'Buys': [", ".join(buys)], 'Sells': [", ".join(sells)]}).to_csv(f'scan_{as_of}.csv', index=False)

    print("\n" + "="*80)
    print(f"1. TECHNICAL ANALYSIS SUMMARY ({as_of})")
    print("="*80)
    print(f"Technically Bullish (100%): {len(buys)} | Accuracy: { (b_win/len(buys)*100) if buys else 0:.2f}%")
    print(f"Technically Bearish (100%): {len(sells)} | Accuracy: { (s_win/len(sells)*100) if sells else 0:.2f}%")
    
    enhanced_req = """HOLDING PERIOD: 0 DTE / End of Market Day. 
MISSION: You are an institutional trader. Filter this TECHNICAL WATCHLIST using:
1. News Sentiment (last 24h) & morning upgrades/downgrades.
2. Sector/Macro Alignment (Check ETF strength vs individual stock news).
3. Catalyst check (Earnings today? CPI?).

OUTPUT: 3 lines (BUY, SELL, NEUTRAL) comma-separated. Include a 1-sentence macro driver summary."""

    print("\n" + "="*80)
    print("2. PROMPT FOR GOOGLE AI (COPY & PASTE)")
    print("="*80)

    # BUY PROMPT
    print("\n[ PROMPT FOR TECHNICAL BUY LIST ]")
    print("-" * 40)
    print(f'TECHNICAL WATCHLIST: {", ".join([f'"{x}"' for x in buys])}\n{enhanced_req}')
    print("-" * 40)
    print(">>> EXECUTION RULE: Trade AI 'BUY' and 'NEUTRAL'. DISCARD AI 'SELL'.")

    # SELL PROMPT
    print("\n[ PROMPT FOR TECHNICAL SELL LIST ]")
    print("-" * 40)
    print(f'TECHNICAL WATCHLIST: {", ".join([f'"{x}"' for x in sells])}\n{enhanced_req}')
    print("-" * 40)
    print(">>> EXECUTION RULE: Trade AI 'SELL' and 'NEUTRAL'. DISCARD AI 'BUY'.")

if __name__ == "__main__":
    run_hybrid_strategy()